# Llama-3.1-8B: Base vs Instruct 비교

두 모델을 각각 로드해서 **동작 방식의 차이**를 직접 확인합니다.

- `meta-llama/Llama-3.1-8B` — **Base(사전학습)**: 주어진 텍스트를 *이어쓰기*(text completion). 지시를 따르지 않고 그럴듯한 다음 토큰을 이어감.
- `meta-llama/Llama-3.1-8B-Instruct` — **Instruct(대화)**: chat template로 지시·대화를 수행.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 107.0 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 58.3 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [3]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_pipe(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return pipeline("text-generation", model=model, tokenizer=tok)

def free(name):
    globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

## 1. Base — 이어쓰기

In [4]:
base = load_pipe("meta-llama/Llama-3.1-8B")

prompt = "AI에 대해 설명하는 글을 작성해."
out = base(
    prompt,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)
print(out[0]["generated_text"])

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'repetition_penalty', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for Wor

AI에 대해 설명하는 글을 작성해.다니지 않습니다. AI가 왜 필요한지, 어떤 식으로 사용되는지를 이해해야 합니다.
AI는 인공지능이 아니라 인공 지능입니다. 머신러닝이란 무엇일까요?
머신 러닝의 원리와 알고리즘은 수학에서 나오는데, 그 수학적 기초를 다뤄야 합니다.

* 학습 목표
1. 머신 러닝의 원리에 대하여 설명한다.
2. 머신 러닝의 알고리즘을 설명하고, 어떻게 적용될 수 있는지를 논한다.
3. 머신 러닝과 �


## 2. Instruct — 대화

In [5]:
free("base")

instruct = load_pipe("meta-llama/Llama-3.1-8B-Instruct")

messages = [
    {"role": "system", "content": "당신은 친절한 한국어 AI 비서입니다."},
    {"role": "user", "content": "AI에 대해 설명하는 글을 작성해."},
]
out = instruct(
    messages,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)
print(out[0]["generated_text"][-1]["content"])

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**인공지능(AI)의 개념과 역사**

인공지능(AI)은 컴퓨터가 인간과 유사한 지능을 가지는 기술을 의미합니다. AI는 컴퓨터가 인간과 유사한 지능을 가지는 기술로, 인간의 지적 노동을 자동화하는 기술입니다. AI는 컴퓨터가 학습, 문제 해결, 자연어 처리, 이미지 인식, 자율 주행 등 다양한 분야에서 인간과 유사한 지능을 가집니다.

AI의 역사

AI의 역사 dates back 1950년대, 그때는 컴퓨터가 인간과 유사한 지능을 가지는 방법을 연구하는 데 집중했습니다. 1950년대부터 1960년대까지, AI는 컴퓨터가 학습, 문제 해결, 자연어 처리 등 다양한 분야에서 인간과 유사한 지능을 가지는 방법을 연구했습니다. 1970년대부터 1980년대까지, AI는 컴퓨터가 인간과 유사한 지능을 가지는 방법을 연구하는 데 집중했습니다.

**인공지능의 종류**

인공지능(AI)은 여러 종류가 있습니다. 그 중에는 다음이 있습니다.


